# GEPA-Style Autonomous Prompt Optimization: Legal Demand Assistant

Autonomous implementation of the GEPA (Genetic-Pareto) methodology:
1. **Evaluate** current prompts against held-out queries
2. **Reflect** — ask the model to diagnose failures from execution traces
3. **Mutate** — ask the model to generate improved prompts
4. **Select** — keep Pareto-optimal candidates
5. **Loop** — repeat for N generations

Fully autonomous — the model evolves its own prompts.

Uses Qwen3.5-9B (open source, no API keys required).

**Requirements:** Colab with GPU (A100 recommended for speed)

## Section 0: Install Dependencies

In [1]:
!pip install transformers>=5.0.0 bitsandbytes>=0.44.0 accelerate

In [5]:
!pip install --upgrade transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 153.7 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


## Section 1: Imports and Model Setup

In [1]:
import json
import copy
from datetime import datetime

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen3.5-9B"
OUTPUT_FILE = "./gepa_optimized_prompts.json"
NUM_GENERATIONS = 3  # Number of autonomous evolution cycles

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

GPU: NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB


In [2]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    attn_implementation="eager",
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
model.eval()
print(f"Model loaded: {MODEL_ID}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/427 [00:00<?, ?it/s]

Model loaded: Qwen/Qwen3.5-9B


## Section 2: Core Functions

In [3]:
def generate(system_prompt, user_input, max_tokens=512, temperature=0.7):
    """Generate a response from the model."""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_input},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_tokens, temperature=temperature,
            top_p=0.9, do_sample=temperature > 0,
        )
    return tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    ).strip()

In [4]:
def score_response(response, criteria, task_type):
    """Score a response against criteria. Returns dict of {criterion: bool} and total score."""
    scores = {}
    resp_lower = response.lower()

    for c in criteria:
        if c == "formal_tone":
            scores[c] = any(w in resp_lower for w in ["dear", "sincerely", "represent", "i represent"])
        elif c == "parties_identified":
            scores[c] = "[" in response or any(w in resp_lower for w in ["my client", "claimant", "tenant", "freelancer"])
        elif c == "amount_stated":
            scores[c] = "$" in response
        elif c == "deadline_included":
            scores[c] = any(w in resp_lower for w in ["days", "deadline", "within"])
        elif c == "escalation_mentioned":
            scores[c] = any(w in resp_lower for w in ["legal action", "legal remedies", "pursue", "file", "court"])
        elif c == "legal_basis":
            scores[c] = any(w in resp_lower for w in ["breach", "violation", "obligation", "liability", "negligence", "statute"])
        elif c == "correct_claim_type":
            scores[c] = any(w in resp_lower for w in ["breach", "fraud", "misrepresentation", "negligence", "violation"])
        elif c == "json_format":
            scores[c] = "{" in response and "}" in response
        elif c == "reasoning_provided":
            scores[c] = any(w in resp_lower for w in ["reasoning", "because", "the ", "failed", "accepted"])
        elif c == "all_fields_extracted":
            scores[c] = all(w in resp_lower for w in ["claimant", "recipient", "damages", "deadline"])
        elif c == "values_correct":
            scores[c] = "$" in response
        elif c == "correctly_identifies_inadequate":
            scores[c] = any(w in resp_lower for w in ["false", "inadequate", "not adequate", "incomplete", "missing"])
        elif c == "correctly_identifies_adequate":
            scores[c] = any(w in resp_lower for w in ["true", "adequate", "effective", "complete", "sufficient"])
        elif c == "missing_elements_listed":
            scores[c] = any(w in resp_lower for w in ["missing", "lacks", "does not include", "no "])
        elif c == "improvements_suggested":
            scores[c] = any(w in resp_lower for w in ["should", "add", "include", "specify", "improve"])
        elif c == "strengths_listed":
            scores[c] = any(w in resp_lower for w in ["strength", "identifies", "includes", "states", "provides"])
        elif c == "multiple_remedies":
            scores[c] = resp_lower.count("remedy") + resp_lower.count("refund") + resp_lower.count("payment") + resp_lower.count("reimbursement") >= 2
        else:
            scores[c] = True

    total = sum(scores.values()) / len(scores) if scores else 0
    return scores, total

In [5]:
def evaluate_prompt(system_prompt, task_type, queries):
    """Run all queries for a task type and return scores + responses."""
    results = []
    for q in queries:
        response = generate(system_prompt, q["input"], max_tokens=256)
        scores, total = score_response(response, q["criteria"], task_type)
        results.append({
            "query_id": q["id"],
            "input": q["input"],
            "response": response,
            "scores": scores,
            "total_score": total,
        })
        print(f"    {q['id']}: {total:.0%} — {scores}")
    avg = sum(r["total_score"] for r in results) / len(results)
    return results, avg

## Section 3: Reflect and Mutate Functions

The core GEPA loop: the model analyzes its own failures and generates improved prompts.

In [6]:
def reflect(system_prompt, results, task_type):
    """Ask the model to analyze what went wrong with the current prompt."""
    traces = ""
    for r in results:
        failed = [c for c, passed in r["scores"].items() if not passed]
        passed = [c for c, p in r["scores"].items() if p]
        traces += (
            f"\nQuery: {r['input'][:150]}...\n"
            f"Response: {r['response'][:300]}...\n"
            f"Passed criteria: {passed}\n"
            f"Failed criteria: {failed}\n"
        )

    reflection_prompt = (
        "You are an expert prompt engineer analyzing system prompt performance.\n\n"
        f"TASK TYPE: {task_type}\n\n"
        f"CURRENT SYSTEM PROMPT:\n{system_prompt}\n\n"
        f"EXECUTION TRACES:\n{traces}\n\n"
        "Analyze the failures above. For each failed criterion, explain WHY the "
        "current system prompt led to that failure. Be specific about what "
        "instruction is missing or unclear. List the top 3 issues to fix, "
        "ordered by impact."
    )

    reflection = generate(
        "You are a prompt engineering expert.",
        reflection_prompt,
        max_tokens=400,
        temperature=0.3,
    )
    return reflection


def mutate(current_prompt, reflection, task_type):
    """Ask the model to generate an improved prompt based on the reflection."""
    mutate_prompt = (
        "You are an expert prompt engineer. Your job is to improve a system prompt "
        "based on a failure analysis.\n\n"
        f"TASK TYPE: {task_type}\n\n"
        f"CURRENT SYSTEM PROMPT:\n{current_prompt}\n\n"
        f"FAILURE ANALYSIS:\n{reflection}\n\n"
        "Write an IMPROVED system prompt that addresses the failures identified above. "
        "Rules:\n"
        "- Output ONLY the new system prompt text, nothing else\n"
        "- Keep it concise (under 300 words)\n"
        "- Be very specific about output format requirements\n"
        "- Include explicit constraints to prevent the identified failures\n"
        "- Do not include any preamble like 'Here is the improved prompt'\n"
        "- Start directly with 'You are...' or the prompt content"
    )

    new_prompt = generate(
        "You are a prompt engineering expert. Output only the requested prompt, nothing else.",
        mutate_prompt,
        max_tokens=500,
        temperature=0.7,
    )

    # Clean up: remove quotes or markdown wrapping if the model added them
    new_prompt = new_prompt.strip('"').strip("'").strip("`").strip()
    if new_prompt.startswith("```"):
        lines = new_prompt.split("\n")
        new_prompt = "\n".join(lines[1:-1] if lines[-1].startswith("```") else lines[1:])

    return new_prompt.strip()

## Section 4: Evaluation Queries

In [7]:
EVAL_QUERIES = {
    "draft_demand_letter": [
        {
            "id": "Q1",
            "input": "Draft a demand letter for a tenant whose landlord withheld a $2,500 security deposit without providing an itemized statement of damages.",
            "criteria": ["formal_tone", "parties_identified", "amount_stated", "deadline_included", "escalation_mentioned", "legal_basis"]
        },
        {
            "id": "Q2",
            "input": "Draft a demand letter for a freelancer who completed branding work for a client, delivered the files, and never received the agreed $3,200 payment.",
            "criteria": ["formal_tone", "parties_identified", "amount_stated", "deadline_included", "escalation_mentioned", "legal_basis"]
        },
    ],
    "identify_claim": [
        {
            "id": "Q3",
            "input": "Identify the strongest legal claim where a contractor accepted an $8,000 payment, performed minimal work, and abandoned the job.",
            "criteria": ["correct_claim_type", "json_format", "reasoning_provided"]
        },
        {
            "id": "Q4",
            "input": "Identify the likely legal claim where a seller knowingly lied about a car having no accident history, and the buyer later discovered major prior damage.",
            "criteria": ["correct_claim_type", "json_format", "reasoning_provided"]
        },
    ],
    "extract_elements": [
        {
            "id": "Q5",
            "input": "Extract the claimant, recipient, damages, deadline, and requested remedy from the following demand letter:\n\nDear Mr. Allen: My client, Sarah Kim, seeks payment of $4,800 for water damage caused to her property on January 8, 2026, when your negligence led to a plumbing overflow from your unit. Please remit payment within 14 days of this letter to avoid further action.",
            "criteria": ["all_fields_extracted", "json_format", "values_correct"]
        },
        {
            "id": "Q6",
            "input": "Extract the key legal elements from the following demand letter:\n\nDear ABC Services, I represent Daniel Ortiz regarding unpaid wages in the amount of $1,950 for work completed during December 2025. Unless payment is received within 7 days, my client will consider further legal action.",
            "criteria": ["all_fields_extracted", "json_format", "values_correct"]
        },
    ],
    "evaluate_letter": [
        {
            "id": "Q7",
            "input": "Evaluate whether this demand letter is complete and effective:\n\nYou owe me money. Please fix this immediately.",
            "criteria": ["correctly_identifies_inadequate", "json_format", "missing_elements_listed", "improvements_suggested"]
        },
        {
            "id": "Q8",
            "input": "Evaluate whether this demand letter is effective:\n\nDear Property Manager, my client seeks return of a $1,800 security deposit for Apartment 2B. The tenant vacated the unit on February 1, 2026, and no itemized deductions were provided. Please return the deposit within 10 days or my client may pursue legal remedies.",
            "criteria": ["correctly_identifies_adequate", "json_format", "strengths_listed"]
        },
    ],
    "recommend_remedy": [
        {
            "id": "Q9",
            "input": "Suggest appropriate remedies when a contractor took a deposit, abandoned a home repair project, and the homeowner had to hire another contractor for additional cost.",
            "criteria": ["multiple_remedies", "json_format", "reasoning_provided"]
        },
        {
            "id": "Q10",
            "input": "Suggest remedies for a consumer who purchased a defective appliance, repeatedly requested repair or refund, and received no response from the seller.",
            "criteria": ["multiple_remedies", "json_format", "reasoning_provided"]
        },
    ],
}

## Section 5: Seed Prompts (Generation 0)

In [8]:
# Generation 0: Minimal seed prompts — the starting point for evolution
seed_prompts = {
    "draft_demand_letter": "You are a helpful legal assistant.",
    "identify_claim": "You are a helpful legal assistant.",
    "extract_elements": "You are a helpful legal assistant.",
    "evaluate_letter": "You are a helpful legal assistant.",
    "recommend_remedy": "You are a helpful legal assistant.",
}

## Section 6: Autonomous Evolution Loop

For each task type, the model:
1. Evaluates the current prompt
2. Reflects on what went wrong
3. Mutates to create an improved prompt
4. Evaluates the new prompt
5. Keeps the better one (selection)
6. Repeats for N generations

In [9]:
# Full evolution history
evolution_history = {task: [] for task in EVAL_QUERIES}
current_prompts = copy.deepcopy(seed_prompts)
current_scores = {}

# Evaluate Generation 0 (baseline)
print("=" * 60)
print("GENERATION 0 — Baseline Evaluation")
print("=" * 60)

gen0_results = {}
for task_type, queries in EVAL_QUERIES.items():
    print(f"\n  [{task_type}]")
    results, avg = evaluate_prompt(current_prompts[task_type], task_type, queries)
    current_scores[task_type] = avg
    gen0_results[task_type] = results
    evolution_history[task_type].append({
        "generation": 0,
        "prompt": current_prompts[task_type],
        "score": avg,
        "reflection": None,
        "action": "seed",
    })

baseline_overall = sum(current_scores.values()) / len(current_scores)
print(f"\nGen 0 Overall: {baseline_overall:.0%}")

Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


GENERATION 0 — Baseline Evaluation

  [draft_demand_letter]


Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


    Q1: 67% — {'formal_tone': True, 'parties_identified': True, 'amount_stated': True, 'deadline_included': True, 'escalation_mentioned': False, 'legal_basis': False}


Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


...

    Q7: 50% — {'correctly_identifies_inadequate': True, 'json_format': False, 'missing_elements_listed': True, 'improvements_suggested': False}


Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


    Q8: 67% — {'correctly_identifies_adequate': True, 'json_format': False, 'strengths_listed': True}

  [recommend_remedy]


Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


    Q9: 33% — {'multiple_remedies': False, 'json_format': False, 'reasoning_provided': True}
    Q10: 67% — {'multiple_remedies': True, 'json_format': False, 'reasoning_provided': True}

Gen 0 Overall: 58%


In [10]:
# Autonomous evolution loop
for gen in range(1, NUM_GENERATIONS + 1):
    print(f"\n{'=' * 60}")
    print(f"GENERATION {gen} — Reflect → Mutate → Evaluate → Select")
    print(f"{'=' * 60}")

    for task_type, queries in EVAL_QUERIES.items():
        print(f"\n  [{task_type}]")
        prev_prompt = current_prompts[task_type]
        prev_score = current_scores[task_type]

        # Get the most recent results for reflection
        prev_results = gen0_results[task_type] if gen == 1 else prev_eval_results.get(task_type, gen0_results[task_type])

        # Step 1: REFLECT — model analyzes its own failures
        print(f"    Reflecting...")
        reflection = reflect(prev_prompt, prev_results, task_type)
        print(f"    Reflection: {reflection[:150]}...")

        # Step 2: MUTATE — model generates improved prompt
        print(f"    Mutating...")
        candidate_prompt = mutate(prev_prompt, reflection, task_type)
        print(f"    New prompt: {candidate_prompt[:100]}...")

        # Step 3: EVALUATE — test the new prompt
        print(f"    Evaluating candidate...")
        candidate_results, candidate_score = evaluate_prompt(candidate_prompt, task_type, queries)

        # Step 4: SELECT — keep the better prompt
        if candidate_score >= prev_score:
            current_prompts[task_type] = candidate_prompt
            current_scores[task_type] = candidate_score
            action = "accepted"
            print(f"    ✓ Accepted: {prev_score:.0%} → {candidate_score:.0%}")
        else:
            action = "rejected"
            print(f"    ✗ Rejected: candidate {candidate_score:.0%} < current {prev_score:.0%}")

        evolution_history[task_type].append({
            "generation": gen,
            "prompt": candidate_prompt,
            "score": candidate_score,
            "prev_score": prev_score,
            "reflection": reflection,
            "action": action,
        })

    # Store results for next generation's reflection
    prev_eval_results = {}
    for task_type, queries in EVAL_QUERIES.items():
        results, _ = evaluate_prompt(current_prompts[task_type], task_type, queries)
        prev_eval_results[task_type] = results

    gen_overall = sum(current_scores.values()) / len(current_scores)
    print(f"\nGen {gen} Overall: {gen_overall:.0%} (Baseline was {baseline_overall:.0%})")

Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.



GENERATION 1 — Reflect → Mutate → Evaluate → Select

  [draft_demand_letter]
    Reflecting...


Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


    Reflection: ### Analysis of System Prompt Failures

The current system prompt (`You are a helpful legal assistant.`) is too generic and lacks specific constraints...
    Mutating...


Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


...

    Q7: 100% — {'correctly_identifies_inadequate': True, 'json_format': True, 'missing_elements_listed': True, 'improvements_suggested': True}


Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


    Q8: 67% — {'correctly_identifies_adequate': False, 'json_format': True, 'strengths_listed': True}


Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


    Q9: 67% — {'multiple_remedies': True, 'json_format': False, 'reasoning_provided': True}
    Q10: 100% — {'multiple_remedies': True, 'json_format': True, 'reasoning_provided': True}

Gen 3 Overall: 88% (Baseline was 58%)


## Section 7: Evolution Summary

In [11]:
print("=" * 60)
print("EVOLUTION SUMMARY")
print("=" * 60)

for task_type in EVAL_QUERIES:
    history = evolution_history[task_type]
    baseline = history[0]["score"]
    final = current_scores[task_type]
    print(f"\n{task_type}:")
    for entry in history:
        gen = entry["generation"]
        score = entry["score"]
        action = entry["action"]
        bar = "█" * int(score * 20) + "░" * (20 - int(score * 20))
        print(f"  Gen {gen}: {bar} {score:.0%} ({action})")
    print(f"  Improvement: {baseline:.0%} → {final:.0%} (Δ {final - baseline:+.0%})")

final_overall = sum(current_scores.values()) / len(current_scores)
print(f"\n{'=' * 60}")
print(f"Overall: {baseline_overall:.0%} → {final_overall:.0%} (Δ {final_overall - baseline_overall:+.0%})")
print(f"{'=' * 60}")

EVOLUTION SUMMARY

draft_demand_letter:
  Gen 0: █████████████░░░░░░░ 67% (seed)
  Gen 1: █████████████░░░░░░░ 67% (accepted)
  Gen 2: █████████████░░░░░░░ 67% (accepted)
  Gen 3: ██████████░░░░░░░░░░ 50% (rejected)
  Improvement: 67% → 67% (Δ +0%)

identify_claim:
  Gen 0: █████████████░░░░░░░ 67% (seed)
  Gen 1: ████████████████████ 100% (accepted)
  Gen 2: ████████████████████ 100% (accepted)
  Gen 3: ████████████████████ 100% (accepted)
  Improvement: 67% → 100% (Δ +33%)

extract_elements:
  Gen 0: ██████████░░░░░░░░░░ 50% (seed)
  Gen 1: ████████████████████ 100% (accepted)
  Gen 2: ████████████████████ 100% (accepted)
  Gen 3: ████████████████████ 100% (accepted)
  Improvement: 50% → 100% (Δ +50%)

evaluate_letter:
  Gen 0: ███████████░░░░░░░░░ 58% (seed)
  Gen 1: ██████████░░░░░░░░░░ 54% (rejected)
  Gen 2: ██████████░░░░░░░░░░ 54% (rejected)
  Gen 3: ██████████████░░░░░░ 71% (accepted)
  Improvement: 58% → 71% (Δ +12%)

recommend_remedy:
  Gen 0: ██████████░░░░░░░░░░ 50% (seed)

## Section 8: View Final Optimized Prompts

In [12]:
for task_type, prompt in current_prompts.items():
    print(f"\n{'='*60}")
    print(f"Task: {task_type} (Score: {current_scores[task_type]:.0%})")
    print(f"{'='*60}")
    print(prompt)


Task: draft_demand_letter (Score: 67%)
You are an expert legal assistant specializing in drafting enforceable demand letters. Your output must strictly adhere to the following mandatory structural and content requirements to ensure legal efficacy:

1.  **Legal Basis Citation**: Explicitly identify and cite the specific statutory authority (e.g., state civil code sections) or contractual clauses governing the dispute. Do not rely on general moral requests; you must state clearly how the recipient's non-compliance violates specific laws or agreements.
2.  **Statutory Consequences**: Explicitly outline the legal consequences of ignoring this demand, including specific statutory penalties, interest accruals, or the right to recover attorney fees and court costs as permitted by the cited law.
3.  **Strict Deadline**: You must calculate and specify a clear, actionable deadline for compliance based on the applicable statute of limitations or notice period. This deadline must be expressed in 

## Section 9: Save Optimized Prompts

In [13]:
output = {
    "metadata": {
        "method": "GEPA-style autonomous prompt optimization",
        "model": MODEL_ID,
        "generations": NUM_GENERATIONS,
        "timestamp": datetime.now().isoformat(),
        "baseline_score": baseline_overall,
        "final_score": final_overall,
    },
    "optimized_prompts": current_prompts,
    "scores": current_scores,
    "evolution_history": evolution_history,
    "seed_prompts": seed_prompts,
}

with open(OUTPUT_FILE, "w") as f:
    json.dump(output, f, indent=2)

print(f"Saved to {OUTPUT_FILE}")

Saved to ./gepa_optimized_prompts.json


## Section 10: Download Results

In [15]:
# Uncomment to download from Colab
from google.colab import files
files.download(OUTPUT_FILE)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>